# CQD-SHAP Fixed Colab Example

This notebook is made for the current CQD-SHAP code. It fixes the old `example_usage.ipynb` problems:

- `setup_dataset_and_graphs(...)` now returns 4 values: `dataset, graph_train, graph_valid, graph_test`.
- `load_query_datasets` is not used anymore.
- `shapley_value` is now called through the `Shapley` class.

Run the cells from top to bottom in Google Colab.

## 1. Use GPU

Before running the notebook, choose:

`Runtime -> Change runtime type -> T4 GPU -> Save`

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone CQD-SHAP and Install Small Dependencies

This clones a fresh copy of the project into `/content/CQD-SHAP`.

In [ ]:
%cd /content
!rm -rf CQD-SHAP
!git clone https://github.com/ds-jrg/CQD-SHAP.git
%cd /content/CQD-SHAP

!pip -q install gdown pandas networkx matplotlib tqdm

## 3. Download Data and Models

These are the Google Drive files used by the original example notebook.

In [ ]:
%cd /content/CQD-SHAP

# Dataset archive. It extracts to data/.
!test -f data.zip || gdown 1yoZFUAY7DLOj4fC78pIU32SUSAEWRmLw -O data.zip
!test -d data || unzip -q data.zip

# Model archive. It extracts to models/.
!test -f models.zip || gdown 1ot3CuVk4DorVu3JiHKzdumzGNaTREAU3 -O models.zip
!test -d models || unzip -q models.zip

## 4. Verify Files

This checks exactly which required files are present, which optional files are present, and which files are missing.

In [ ]:
from pathlib import Path

required_paths = {
    "main script": "evaluation.py",
    "benchmark 1 data": "data/FB15k-237",
    "Freebase model": "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt",
}

optional_paths = {
    "benchmark 2 +H data": "data/FB15k-237+H",
    "NELL data": "data/NELL995",
    "NELL +H data": "data/NELL995+H",
    "NELL model": "models/NELL-model-rank-1000-epoch-100-1602499096.pt",
}

missing_required = []

print("Required files/folders:")
for label, path in required_paths.items():
    exists = Path(path).exists()
    status = "FOUND" if exists else "MISSING"
    print(f"- {label}: {status} -> {path}")
    if not exists:
        missing_required.append(path)

print("\nOptional files/folders:")
for label, path in optional_paths.items():
    exists = Path(path).exists()
    status = "FOUND" if exists else "NOT FOUND"
    print(f"- {label}: {status} -> {path}")

if missing_required:
    raise FileNotFoundError("Missing required paths: " + ", ".join(missing_required))

print("\nDecision:")
if Path("data/FB15k-237+H").exists():
    print("- Notebook will use benchmark 2: data/FB15k-237+H")
else:
    print("- Notebook will use benchmark 1: data/FB15k-237")

## 4.1 Show Current Folder Contents

Use this when you need to report what Colab currently has.

In [ ]:
!echo "Project root:"
!find . -maxdepth 1 -mindepth 1 | sort
!echo "\nData folders:"
!find data -maxdepth 2 -type d 2>/dev/null | sort || true
!echo "\nModel files:"
!find models -maxdepth 1 -type f 2>/dev/null | sort || true

## 5. Import Current Code

This import cell matches the current repository code.

In [ ]:
from symbolic_torch import SymbolicReasoning
from xcqa_torch import XCQA
from utils import setup_dataset_and_graphs, load_all_queries, get_num_atoms, compute_rank, human_readable
from shapley import Shapley

import random
random.seed(42)
torch.manual_seed(42)

print("Imports OK")

## 6. Load Freebase Data

If `data/FB15k-237+H` exists, the notebook uses benchmark 2. Otherwise it uses `data/FB15k-237` with benchmark 1 settings.

In [ ]:
if Path("data/FB15k-237+H").exists():
    data_dir = "data/FB15k-237+H"
    benchmark_version = 2
    add_reverse = False
else:
    data_dir = "data/FB15k-237"
    benchmark_version = 1
    add_reverse = True

model_path = "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt"
print("Using data_dir:", data_dir)
print("Using benchmark_version:", benchmark_version)

dataset, graph_train, graph_valid, graph_test = setup_dataset_and_graphs(
    data_dir,
    logging=True,
    add_reverse=add_reverse,
)

query_dataset, query_dataset_hard = load_all_queries(
    dataset,
    data_dir,
    split="test",
    version=benchmark_version,
)

print("Dataset and queries loaded")

## 7. Load the XCQA Model

In [ ]:
reasoner = SymbolicReasoning(graph_valid, logging=False)
xcqa = XCQA(
    symbolic=reasoner,
    dataset=dataset,
    logging=False,
    model_path=model_path,
    normalize=False,
)

print("Model loaded")

## 8. Pick One Query and Run CQD

In [ ]:
query_type = "2p"
k = 10
t_norm = "prod"
t_conorm = "prod"

queries_hard = query_dataset_hard.get_queries(query_type)
queries_complete = query_dataset.get_queries(query_type)

query_idx = None
for idx, query in enumerate(queries_hard):
    if len(query.answer) > 0:
        query_idx = idx
        break

assert query_idx is not None, f"No hard-answer query found for {query_type}"

query_hard = queries_hard[query_idx]
query_complete = queries_complete[query_idx]
num_atoms = get_num_atoms(query_type)
all_answers = set(query_complete.get_answer())
target = query_hard.answer[0]

print("Query type:", query_type)
print("Query index:", query_idx)
print("Target answer:", target, dataset.get_title_by_id(target))
print("Human-readable query:")
print(human_readable(query_hard, dataset))

In [ ]:
full_coalition = [1] * num_atoms

result = xcqa.query_execution(
    query_hard,
    k=k,
    coalition=full_coalition,
    t_norm=t_norm,
    t_conorm=t_conorm,
)

rank = compute_rank(result, query_complete.answer, target)
print("Filtered rank of target:", rank)
print("Top 10 predicted answers:")

for i, entity_id in enumerate(result.index[:10], start=1):
    title = dataset.get_title_by_id(entity_id)
    print(f"{i}. {entity_id}: {title}")

## 9. Compute CQD-SHAP Values for the Query

In [ ]:
shapley = Shapley(
    xcqa,
    qoi="rank",
    k=k,
    t_norm=t_norm,
    t_conorm=t_conorm,
)

filtered_exclude = all_answers - {target}
shapley_values = shapley.shapley_values(query_hard, filtered_exclude, target)

print("Shapley values:")
for atom, value in shapley_values.items():
    print(f"Atom {atom}: {value}")

best_atom = max(shapley_values, key=shapley_values.get)
print("Most important atom:", best_atom)

## 10. Run Evaluation Commands

The first command is a smaller test on one query type. The full experiment cell runs all methods and can take much longer.

In [ ]:
# Quick evaluation test for one query type.
!python evaluation.py --kg Freebase --benchmark {benchmark_version} --method shapley --query_type 2p --data_dir {data_dir} --model_path {model_path}

In [ ]:
# Full Freebase benchmark runs for all methods.
# Change RUN_FULL_EXPERIMENT to True when you need complete report results.

RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    import subprocess
    methods = ["shapley", "score", "random", "first", "last"]
    output_dir = f"evaluation_benchmark{benchmark_version}/Freebase"
    print("Output folder:", output_dir)
    for method in methods:
        print("Running method:", method)
        subprocess.run([
            "python", "evaluation.py",
            "--kg", "Freebase",
            "--benchmark", str(benchmark_version),
            "--method", method,
            "--data_dir", data_dir,
            "--model_path", model_path,
        ], check=True)
else:
    print("Full experiment skipped.")
    print("Set RUN_FULL_EXPERIMENT = True, then rerun this cell to generate all method outputs.")
    print("Methods that will run: shapley, score, random, first, last")

## 10.1 Exact Full Experiment Commands

These commands are printed so you can paste them into a report or run them manually.

In [ ]:
methods = ["shapley", "score", "random", "first", "last"]
for method in methods:
    command = (
        f"python evaluation.py --kg Freebase --benchmark {benchmark_version} "
        f"--method {method} --data_dir {data_dir} --model_path {model_path}"
    )
    print(command)

## 11. Find Output Files

Evaluation output is saved under `evaluation_benchmark<benchmark_version>/Freebase/`.

In [ ]:
!find evaluation_benchmark{benchmark_version} -maxdepth 3 -type f 2>/dev/null | sort || true